In [ ]:
import torch
from torch import nn


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )

    def forward(self, x):
        return self.convs(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1):
        super().__init__()

        self.down1 = ConvBlock(in_ch, 64)
        self.down2 = ConvBlock(64, 128)
        self.bot1 = ConvBlock(128, 256)
        self.up2 = ConvBlock(128 + 256, 128)
        self.up1 = ConvBlock(128 + 64, 64)
        self.out = nn.Conv2d(64, in_ch, 1)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear') # 쌍선형 보간법. 이미지 크기를 확대할 때 사용

    def forward(self, x):               # (N, 1, 28, 28)
        x1 = self.down1(x)              # (N, 64, 28, 28)   채널을 1 -> 64로 확장
        x = self.maxpool(x1)            # (N, 64, 14, 14)   크기를 절반

        x2 = self.down2(x)              # (N, 128, 14, 14)  채널을 64 -> 128로 확장
        x = self.maxpool(x2)            # (N, 128, 7, 7)    크기를 절반

        x = self.bot1(x)                # (N, 256, 7, 7)    채널을 128 -> 256으로 확장

        x = self.upsample(x)            # (N, 256, 14, 14)  이미지 크기를 2배 확대
        x = torch.cat([x, x2], dim=1)   # (N, 256 + 128, 14, 14) 스킵 연결로 채널을 합침. (dim=1 : 채널 축)

        # 스킵 연결은 maxpool에서 잃은 세밀한 위치 정보를, 같은 해상도의 인코더 특징 지도에서 디코더로 직접 넘겨주는 기법
        
        x = self.up2(x)                 # (N, 128, 14, 14)  채널을 384 -> 128로 축소
        x = self.upsample(x)            # (N, 128, 28, 28)  이미지 크기를 2배 확대
        x = torch.cat([x, x1], dim=1)   # (N, 128 + 64, 28, 28)  스킵 연결로 채널을 합침. (dim=1 : 채널 축)
        x = self.up1(x)                 # (N, 64, 28, 28)   채널을 192 -> 64로 축소
        x = self.out(x)                 # (N, 1, 28, 28)    채널을 64 -> 1로 축소
        return x


model = UNet()
x = torch.randn(10, 1, 28, 28)
y = model(x)
print(y.shape) # (10, 1, 28, 28)

torch.Size([10, 1, 28, 28])
